In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import spikeinterface as si
import matplotlib.pyplot as plt
import os
from matplotlib.backends.backend_pdf import PdfPages

from tqdm import tqdm


import sys
import spikeinterface as si
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre


import torch.nn.functional as F
from pathlib import Path


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
import time
import pickle
import networkx as nx
from sklearn.metrics import accuracy_score
# from function.Function import *

In [2]:
def count_array2_in_range_of_array1(array1, array2, threshold=5):

    sorted_array1 = np.sort(array1)
    array2 = np.sort(array2)
    
    lefts = array2 - threshold
    rights = array2 + threshold
    
    left_indices = np.searchsorted(sorted_array1, lefts, side='left')
    
    right_indices = np.searchsorted(sorted_array1, rights, side='right')
    
    has_within_range = right_indices > left_indices
    
    count = np.sum(has_within_range)
    
    return count

def label_array1_based_on_array2(array1, array2, threshold=5):
    array_1 = np.sort(array1)
    sorted_array2 = np.sort(array2)
    
    labels = np.zeros(len(array1), dtype=int)
    
    for i, value in enumerate(array1):
        left = value - threshold
        right = value + threshold
        
        left_index = np.searchsorted(sorted_array2, left, side='left')
        right_index = np.searchsorted(sorted_array2, right, side='right')
        
        if right_index > left_index:
            labels[i] = 1
    
    return labels
def detect_local_maxima_in_window(data, window_size=20, std_multiplier=2):

    """
    在每个滑动窗口范围内检测局部最大值的索引，并确保最大值大于两倍的标准差。

    参数:
    data : numpy.ndarray
        输入数据，形状为 (n_rows, n_columns)。
    window_size : int
        滑动窗口的大小，用于定义局部范围，默认为 20。
    std_multiplier : float
        标准差的倍数，用于筛选局部最大值，默认为 2。

    返回:
    local_maxima_indices : list of numpy.ndarray
        每行局部最大值的索引列表，每个元素是对应行局部最大值的索引数组。
    """
    local_maxima_indices = []

    for row in data:
        maxima_indices = []
        row_std = np.std(row.astype(np.float32))
        threshold = std_multiplier * row_std

        for start in range(0, len(row), window_size):
            end = min(start + window_size, len(row))
            window = np.abs(row[start:end])
            
            if len(window) > 0:
                local_max_index = np.argmax(window)
                local_max_value = window[local_max_index]
                
                if local_max_value > threshold:
                    maxima_indices.append(start + local_max_index)  
        
        local_maxima_indices.extend(maxima_indices)
        local_maxima_indices = list(set(local_maxima_indices))  

    return local_maxima_indices
def cluster_label_array1_based_on_array2(array1, array2, threshold=5):

    """
    根据 array2 的 'time' 和 'cluster' 对 array1 进行标记。
    如果 array1 中的某个值在 threshold 范围内存在于 array2 的 'time' 中，则标记为对应的 'cluster' 值，否则为 0。
    
    参数:
    array1 : numpy.ndarray
        要标记的数组。
    array2 : numpy.ndarray
        包含 'time' 和 'cluster' 的二维数组。
        第一列为 'time'，第二列为 'cluster'。
    threshold : int
        判断范围的阈值。
    
    返回:
    labels : numpy.ndarray
        长度为 len(array1) 的标签数组，值为 array2 中的 'cluster' 或 0。
    """

    array2 = np.array(array2.iloc[:, [5, 1]])
    sorted_indices = np.argsort(array2[:, 0])
    sorted_array2 = array2[sorted_indices]
    
    labels = np.zeros(len(array1), dtype=int)
    
    # 遍历 array1 中的每个元素
    for i, value in enumerate(array1):
        # 计算当前值的范围
        left = value - threshold
        right = value + threshold
        
        left_index = np.searchsorted(sorted_array2[:, 0], left, side='left')
        right_index = np.searchsorted(sorted_array2[:, 0], right, side='right')
        
        # 如果范围内存在值，则标记为对应的 'cluster'
        if right_index > left_index:
            # 获取范围内的第一个匹配值的 'cluster'
            labels[i] = sorted_array2[left_index, 1]
    
    return labels
def label_array1_based_on_array2(array1, array2, threshold=5):

    """
    根据 array2 的值对 array1 进行标记。
    如果 array1 中的某个值在 threshold 范围内存在于 array2 中，则标记为 1，否则为 0。
    
    参数:
    array1 : numpy.ndarray
        要标记的数组。
    array2 : numpy.ndarray
        用于判断的数组。
    threshold : int
        判断范围的阈值。
    
    返回:
    labels : numpy.ndarray
        长度为 len(array1) 的标签数组，值为 0 或 1。
    """
    # 对 array2 进行排序以加速搜索
    sorted_array2 = np.sort(array2)
    
    # 初始化标签数组，默认值为 0
    labels = np.zeros(len(array1), dtype=int)
    
    # 遍历 array1 中的每个元素
    for i, value in enumerate(array1):
        # 计算当前值的范围
        left = value - threshold
        right = value + threshold
        
        # 使用二分搜索判断范围内是否存在值
        left_index = np.searchsorted(sorted_array2, left, side='left')
        right_index = np.searchsorted(sorted_array2, right, side='right')
        
        # 如果范围内存在值，则标记为 1
        if right_index > left_index:
            labels[i] = 1
    
    return labels
def extract_windows(data, indices, window_size=61):
    """
    根据给定的时间点索引提取窗口。
    
    参数:
    data : numpy.ndarray
        输入数据，形状为 (n_channels, time)
    indices : numpy.ndarray
        时间点索引数组，用于指定需要提取窗口的中心点
    window_size : int
        窗口长度，默认为61（对应time-30到time+31）
    
    返回:
    windows : numpy.ndarray
        提取的窗口数据，形状为 (len(indices), n_channels, window_size)
    """
    n_channels, time_length = data.shape
    half_window = window_size // 2

    if np.any(indices < half_window) or np.any(indices >= time_length - half_window):
        raise ValueError("Some indices are out of bounds for the given window size.")

    windows = []
    for idx in indices:
        window = data[:, idx - half_window:idx + half_window + 1]
        windows.append(window)

    windows = np.array(windows)
    return windows

In [3]:
recording_raw = se.MEArecRecordingExtractor(file_path='/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_type1.h5')
probe_384channel = recording_raw.get_probegroup()

In [4]:
recording_raw = se.read_binary(file_paths='/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/03_Neuropixel_384_channels_visual_stimuli/raw_data/810755797/spike_band.dat', sampling_frequency=30000, dtype=np.int16, num_channels=384)
recording_raw = recording_raw.set_probegroup(probe_384channel)
recording_f = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
recording_f = spre.common_reference(recording_f, reference="global", operator="median")
recording_f = recording_f.time_slice(start_time=0, end_time= 1200)


In [5]:
probe_384channel = np.array(recording_raw.get_probegroup().to_dataframe().iloc[:, 1:3])

In [6]:
eps=1e-5
distance_threshold = 100
dist_matrix = np.linalg.norm(probe_384channel[:, np.newaxis] - probe_384channel, axis=2)
np.fill_diagonal(dist_matrix, 0)
dist_matrix[dist_matrix < eps] = eps
inv_dist = np.zeros_like(dist_matrix)

inv_dist = np.where(dist_matrix > 0, 1, 0)
np.fill_diagonal(inv_dist, 0)  
if distance_threshold is not None:
    inv_dist[dist_matrix > distance_threshold] = 0

graph = nx.from_numpy_array(inv_dist)
maximal_cliques = list(nx.find_cliques(graph))
cliques_dict = {i: clique for i, clique in enumerate(maximal_cliques)}

In [7]:
spike_inf = pd.read_csv("/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/03_Neuropixel_384_channels_visual_stimuli/spike_sorting/810755797/spike_inf.tsv", index_col=0, sep='\t')
cluster_inf = pd.read_csv("/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/03_Neuropixel_384_channels_visual_stimuli/spike_sorting/810755797/cluster_inf.csv", index_col=0)

In [8]:
class CustomDataset(Dataset):
    def __init__(self, data, labels):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]
    



class Spike_Classification_MLP(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, num_classes, window_size, channel_num):
        super(Spike_Classification_MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size2, num_classes)  
        self.window_size = window_size
        self.channel_num = channel_num

    def forward(self, x):
        x = x.reshape(-1, self.window_size * self.channel_num)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)  
        return x

In [9]:
seen_results = {}  
unique_i_list = [] 

for i in sorted(cliques_dict.keys()):  
    probe_center = np.mean(probe_384channel[cliques_dict[i]], axis=0) 
    distance_threshold = 100  
    
    distances = np.linalg.norm(cluster_inf.iloc[:, [-2, -1]] - probe_center, axis=1)
    indices_within_range = cluster_inf['cluster_id'].values[np.where(distances <= distance_threshold)[0]]
    if len(indices_within_range) != 0:
        result_tuple = tuple(sorted(indices_within_range))  
        
        if result_tuple not in seen_results:
            seen_results[result_tuple] = i  
            unique_i_list.append(i)       


In [10]:
# Spike Classification Training

print("=" * 80)
print("开始 Spike Classification 训练")
print("=" * 80)

classification_accuracy_dict = {}
single_cluster_mapping = {}  # 存储只有一个cluster的clique的映射

# ============================================================================
# 阶段1：预处理 - 确定每个clique对应的clusters和spike信息
# ============================================================================
print("\n阶段1：预处理clique和cluster信息...")

clique_cluster_mapping = {}  # {clique_id: {'clusters': [...], 'spike_data': DataFrame}}

for clique_id in unique_i_list:
    clique = cliques_dict[clique_id]
    probe_center = np.mean(probe_384channel[clique], axis=0) 
    distance_threshold = 100  

    distances = np.linalg.norm(cluster_inf.iloc[:, [-2, -1]] - probe_center, axis=1)
    indices_within_range = cluster_inf['cluster_id'].values[np.where(distances <= distance_threshold)[0]]
    
    if len(indices_within_range) == 0:
        continue
    
    spike_inf_clique = spike_inf[spike_inf['cluster'].isin(indices_within_range)]
    unique_clusters = np.unique(spike_inf_clique['cluster'])
    
    clique_cluster_mapping[clique_id] = {
        'clusters': unique_clusters,
        'cluster_indices': indices_within_range,
        'spike_data': spike_inf_clique,
        'clique': clique
    }

print(f"  完成预处理，共 {len(clique_cluster_mapping)} 个有效cliques")

# ============================================================================
# 阶段2：一次性提取所有需要的spike窗口
# ============================================================================
print("\n阶段2：一次性提取所有spike窗口...")

total_time = 1200 * 30000
train_time = int(total_time * 0.8)
chunk_size = 120000
window_size = 31
half_window = window_size // 2

# 收集所有需要的spike时间点及其对应的clique和cluster信息
all_spike_info = []
for clique_id, info in clique_cluster_mapping.items():
    spike_data = info['spike_data'].copy()
    spike_data['clique_id'] = clique_id
    all_spike_info.append(spike_data)

if len(all_spike_info) > 0:
    all_spike_info = pd.concat(all_spike_info, axis=0, ignore_index=True)
    print(f"  总计需要提取 {len(all_spike_info)} 个spikes的窗口")
    
    # 提取训练集的窗口
    print("  提取训练集窗口...")
    train_spike_windows = {}  # {(time, clique_id): window}
    
    train_spikes = all_spike_info[all_spike_info['time'] < train_time]
    
    for start_frame in tqdm(range(0, train_time, chunk_size), desc="  训练集"):
        end_frame = min(start_frame + chunk_size, train_time)
        
        spikes_in_chunk = train_spikes[
            (train_spikes['time'] >= start_frame + half_window + 1) & 
            (train_spikes['time'] < end_frame - half_window)
        ]
        
        if len(spikes_in_chunk) == 0:
            continue
        
        # 获取这个chunk中涉及的所有cliques
        cliques_in_chunk = spikes_in_chunk['clique_id'].unique()
        
        # 读取所有需要的channels（所有涉及的cliques的channels的并集）
        all_channels = set()
        for cid in cliques_in_chunk:
            all_channels.update(clique_cluster_mapping[cid]['clique'])
        all_channels = sorted(list(all_channels))
        
        data_chunk = recording_f.get_traces(
            start_frame=start_frame,
            end_frame=end_frame,
            channel_ids=[i for i in all_channels]
        )
        
        # 为每个spike提取窗口
        for _, spike_row in spikes_in_chunk.iterrows():
            spike_time = int(spike_row['time'])  # 确保是整数
            clique_id = spike_row['clique_id']
            clique_channels = clique_cluster_mapping[clique_id]['clique']
            
            # 找到这些channels在data_chunk中的索引
            channel_indices = [all_channels.index(ch) for ch in clique_channels]
            
            rel_idx = int(spike_time - start_frame)  # 确保索引是整数
            window = data_chunk.T[channel_indices, rel_idx-half_window : rel_idx+half_window+1]
            
            key = (spike_time, clique_id, spike_row['cluster'])
            train_spike_windows[key] = window
    
    print(f"  训练集: 提取了 {len(train_spike_windows)} 个窗口")
    
    # 提取验证集的窗口
    print("  提取验证集窗口...")
    val_spike_windows = {}  # {(time, clique_id): window}
    
    val_spikes = all_spike_info[all_spike_info['time'] >= train_time]
    
    for start_frame in tqdm(range(train_time, total_time, chunk_size), desc="  验证集"):
        end_frame = min(start_frame + chunk_size, total_time)
        
        spikes_in_chunk = val_spikes[
            (val_spikes['time'] >= start_frame + half_window + 1) & 
            (val_spikes['time'] < end_frame - half_window)
        ]
        
        if len(spikes_in_chunk) == 0:
            continue
        
        # 获取这个chunk中涉及的所有cliques
        cliques_in_chunk = spikes_in_chunk['clique_id'].unique()
        
        # 读取所有需要的channels
        all_channels = set()
        for cid in cliques_in_chunk:
            all_channels.update(clique_cluster_mapping[cid]['clique'])
        all_channels = sorted(list(all_channels))
        
        data_chunk = recording_f.get_traces(
            start_frame=start_frame,
            end_frame=end_frame,
            channel_ids=[i for i in all_channels]
        )
        
        # 为每个spike提取窗口
        for _, spike_row in spikes_in_chunk.iterrows():
            spike_time = int(spike_row['time'])  # 确保是整数
            clique_id = spike_row['clique_id']
            clique_channels = clique_cluster_mapping[clique_id]['clique']
            
            # 找到这些channels在data_chunk中的索引
            channel_indices = [all_channels.index(ch) for ch in clique_channels]
            
            rel_idx = int(spike_time - start_frame)  # 确保索引是整数
            window = data_chunk.T[channel_indices, rel_idx-half_window : rel_idx+half_window+1]
            
            key = (spike_time, clique_id, spike_row['cluster'])
            val_spike_windows[key] = window
    
    print(f"  验证集: 提取了 {len(val_spike_windows)} 个窗口")
else:
    print("  没有需要提取的spike窗口")
    train_spike_windows = {}
    val_spike_windows = {}

# ============================================================================
# 阶段3：对每个clique训练模型
# ============================================================================
print("\n阶段3：训练分类模型...")
print("=" * 80)

for clique_id in clique_cluster_mapping.keys():
    print(f'\n处理 Clique {clique_id}...')
    os.makedirs(f'/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/03_Neuropixel_384_channels_visual_stimuli/spike_classification/train_result/810755797/{clique_id}', exist_ok=True)
    os.makedirs(f'/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/03_Neuropixel_384_channels_visual_stimuli/spike_classification/eval_result/810755797/{clique_id}', exist_ok=True)
    
    classification_accuracy_dict[clique_id] = []
    
    # 从预处理的映射中获取信息
    clique_info = clique_cluster_mapping[clique_id]
    spike_inf_clique = clique_info['spike_data']
    unique_clusters = clique_info['clusters']
    
    # 创建cluster到index的映射
    cluster_to_index = {cluster: idx for idx, cluster in enumerate(unique_clusters)}
    
    print(f"  Cluster数量: {len(unique_clusters)}, Cluster IDs: {list(unique_clusters)}")
    
    # 检测是否只有一个cluster
    if len(unique_clusters) == 1:
        single_cluster_id = unique_clusters[0]
        original_cluster_id = list(cluster_to_index.keys())[0]
        single_cluster_mapping[clique_id] = {
            'cluster_id': single_cluster_id,
            'original_cluster_id': original_cluster_id,
            'n_spikes': len(spike_inf_clique)
        }
        print(f"  ⚠️  只有1个cluster (原始ID: {original_cluster_id})，跳过训练")
        print(f"  📌 已创建映射：Clique {clique_id} 的所有spike → Cluster {original_cluster_id}")
        
        # 保存映射信息
        with open(f"/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/03_Neuropixel_384_channels_visual_stimuli/spike_classification/eval_result/810755797/{clique_id}/single_cluster_mapping.pkl", 'wb') as f:
            pickle.dump(single_cluster_mapping[clique_id], f)
        
        continue
    
    # 从预提取的窗口中获取该clique的训练数据
    print("  从预提取的窗口中获取训练数据...")
    train_windows_list = []
    train_labels_list = []
    
    for key, window in train_spike_windows.items():
        spike_time, cid, cluster = key  # 改用spike_time避免覆盖time模块
        if cid == clique_id:
            train_windows_list.append(window)
            train_labels_list.append(cluster_to_index[cluster])
    
    if len(train_windows_list) == 0:
        print(f"  Clique {clique_id}: 没有训练数据，跳过")
        continue
    
    all_windows = np.stack(train_windows_list)
    labels = np.array(train_labels_list)
    
    print(f"  训练集: {len(all_windows)} 个样本")
    
    # 平衡数据集
    balanced_indices = []
    for cluster_idx in range(len(unique_clusters)):
        cluster_indices = np.where(labels == cluster_idx)[0]
        if len(cluster_indices) > 8000:
            sampled_indices = np.random.choice(cluster_indices, 8000, replace=False)
        else:
            sampled_indices = cluster_indices
        balanced_indices.extend(sampled_indices)
    
    np.random.shuffle(balanced_indices)
    
    balanced_data = all_windows[balanced_indices]
    balanced_labels = labels[balanced_indices]
    
    dataset = CustomDataset(balanced_data, balanced_labels)
    
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    
    batch_size = 1024
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    
    input_size = balanced_data.shape[1] * balanced_data.shape[2]
    num_classes = len(unique_clusters)
    
    # 从预提取的窗口中获取验证数据
    print("  从预提取的窗口中获取验证数据...")
    val_windows_list = []
    val_labels_list = []
    
    for key, window in val_spike_windows.items():
        spike_time, cid, cluster = key  # 改用spike_time避免覆盖time模块
        if cid == clique_id:
            val_windows_list.append(window)
            val_labels_list.append(cluster_to_index[cluster])
    
    if len(val_windows_list) == 0:
        print(f"  Clique {clique_id}: 没有验证数据，跳过")
        continue
    
    all_windows_val = np.stack(val_windows_list)
    labels_val = np.array(val_labels_list)
    
    print(f"  验证集: {len(all_windows_val)} 个样本")
    
    dataset_val = CustomDataset(all_windows_val, labels_val)
    val_loader = DataLoader(dataset_val, batch_size=batch_size, shuffle=False, num_workers=0)
    
    # 训练模型
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    hidden_size1 = 64
    hidden_size2 = 50
    channel_num = balanced_data.shape[1]
    
    for trail in range(1, 6):
        print(f"  Trail {trail}...")
        
        model = Spike_Classification_MLP(input_size, hidden_size1, hidden_size2, num_classes, 
                                        window_size=window_size, channel_num=channel_num)
        model = model.to(device)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.00001)
        
        num_epochs = 210
        accuracy_best = 0
        i = 0
        
        for epoch in range(num_epochs):
            # 训练阶段
            model.train()
            total_loss = 0
            all_labels = []
            all_predictions = []
            
            for batch_data, batch_labels in train_loader:
                batch_data = batch_data.to(device)
                batch_labels = batch_labels.to(device)
                
                outputs = model(batch_data)
                predicted = torch.argmax(outputs, dim=1)
                
                loss = criterion(outputs, batch_labels)
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
                all_labels.extend(batch_labels.cpu().numpy())
                all_predictions.extend(predicted.cpu().numpy())
            
            train_accuracy = accuracy_score(all_labels, all_predictions)
            
            # 验证阶段
            model.eval()
            all_labels = []
            all_predictions = []
            
            with torch.no_grad():
                for batch_data, batch_labels in val_loader:
                    batch_data = batch_data.to(device)
                    batch_labels = batch_labels.to(device)
                    
                    outputs = model(batch_data)
                    predicted = torch.argmax(outputs, dim=1)
                    
                    all_labels.extend(batch_labels.cpu().numpy())
                    all_predictions.extend(predicted.cpu().numpy())
            
            all_labels = np.array(all_labels)
            all_predictions = np.array(all_predictions)
            
            val_accuracy = accuracy_score(all_labels, all_predictions)
            
            if val_accuracy > accuracy_best:
                accuracy_best = val_accuracy
                i = 0
                torch.save(model, f'/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/03_Neuropixel_384_channels_visual_stimuli/spike_classification/train_result/810755797/{clique_id}/spike_classification_model_{trail}.pth')
                if epoch % 10 == 0:
                    print(f"Epoch {epoch}: Best model saved with Accuracy: {accuracy_best:.4f}")
            else:
                i += 1
                if i == 3:
                    print(f"Training stopped at epoch {epoch+1} with best Accuracy: {accuracy_best:.4f}")
                    classification_accuracy_dict[clique_id].append(accuracy_best)
                    break
        
        with open(f"/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/03_Neuropixel_384_channels_visual_stimuli/spike_classification/eval_result/810755797/{clique_id}/accuracy_{trail}.pkl", 'wb') as f:
            pickle.dump(classification_accuracy_dict[clique_id], f)
        

print("\n" + "=" * 80)
print("Spike Classification 训练完成!")
print("=" * 80)


开始 Spike Classification 训练

阶段1：预处理clique和cluster信息...
  完成预处理，共 127 个有效cliques

阶段2：一次性提取所有spike窗口...
  总计需要提取 22731630 个spikes的窗口
  提取训练集窗口...


  训练集: 100%|██████████| 240/240 [17:52<00:00,  4.47s/it]


  训练集: 提取了 19623436 个窗口
  提取验证集窗口...


  验证集: 100%|██████████| 60/60 [03:13<00:00,  3.23s/it]


  验证集: 提取了 3101861 个窗口

阶段3：训练分类模型...

处理 Clique 3...
  Cluster数量: 1, Cluster IDs: [np.int64(0)]
  ⚠️  只有1个cluster (原始ID: 0)，跳过训练
  📌 已创建映射：Clique 3 的所有spike → Cluster 0

处理 Clique 5...
  Cluster数量: 10, Cluster IDs: [np.int64(0), np.int64(2), np.int64(3), np.int64(4), np.int64(6), np.int64(10), np.int64(243), np.int64(244), np.int64(245), np.int64(246)]
  从预提取的窗口中获取训练数据...
  训练集: 187065 个样本
  从预提取的窗口中获取验证数据...
  验证集: 34507 个样本
  Trail 1...
Epoch 0: Best model saved with Accuracy: 0.2119
Training stopped at epoch 4 with best Accuracy: 0.2119
  Trail 2...
Epoch 0: Best model saved with Accuracy: 0.0809
Epoch 10: Best model saved with Accuracy: 0.2064
Epoch 20: Best model saved with Accuracy: 0.2371
Epoch 30: Best model saved with Accuracy: 0.2527
Epoch 40: Best model saved with Accuracy: 0.2627
Epoch 50: Best model saved with Accuracy: 0.2735
Epoch 70: Best model saved with Accuracy: 0.2896
Epoch 80: Best model saved with Accuracy: 0.2981
Epoch 90: Best model saved with Accuracy: 0.3043


KeyboardInterrupt: 

In [ ]:
# 保存Spike Classification结果
print("保存 Spike Classification 结果...")

# 保存准确率字典
with open("/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/03_Neuropixel_384_channels_visual_stimuli/spike_classification/eval_result/810755797/classification_accuracy_dict.pkl", 'wb') as f:
    pickle.dump(classification_accuracy_dict, f)

# 保存单cluster映射
with open("/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/03_Neuropixel_384_channels_visual_stimuli/spike_classification/eval_result/810755797/single_cluster_mapping.pkl", 'wb') as f:
    pickle.dump(single_cluster_mapping, f)

# 打印结果摘要
print("\n" + "=" * 80)
print("Spike Classification 结果摘要")
print("=" * 80)

print("\n多Cluster Cliques (已训练模型):")
for clique_id, accuracies in classification_accuracy_dict.items():
    if len(accuracies) > 0:
        mean_acc = np.mean(accuracies)
        std_acc = np.std(accuracies)
        print(f"  Clique {clique_id}: Mean Accuracy = {mean_acc:.4f} ± {std_acc:.4f}, Trials = {len(accuracies)}")

print(f"\n单Cluster Cliques (直接映射，共 {len(single_cluster_mapping)} 个):")
for clique_id, mapping in single_cluster_mapping.items():
    print(f"  Clique {clique_id}: → Cluster {mapping['original_cluster_id']} (n_spikes = {mapping['n_spikes']})")

print(f"\n统计:")
print(f"  - 训练模型的Cliques: {len([acc for acc in classification_accuracy_dict.values() if len(acc) > 0])} 个")
print(f"  - 单Cluster Cliques: {len(single_cluster_mapping)} 个")
print(f"  - 总计: {len([acc for acc in classification_accuracy_dict.values() if len(acc) > 0]) + len(single_cluster_mapping)} 个")

print("\n所有结果已保存到:")
print("  - 模型文件: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/03_Neuropixel_384_channels_visual_stimuli/spike_classification/train_result/810755797/")
print("  - 评估结果: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/03_Neuropixel_384_channels_visual_stimuli/spike_classification/eval_result/810755797/")
print("  - 单Cluster映射: single_cluster_mapping.pkl")
print("=" * 80)


保存 Spike Classification 结果...

Spike Classification 结果摘要

多Cluster Cliques (已训练模型):
  Clique 5: Mean Accuracy = 0.8445 ± 0.0061, Trials = 5
  Clique 6: Mean Accuracy = 0.6923 ± 0.2233, Trials = 5
  Clique 14: Mean Accuracy = 0.9587 ± 0.0040, Trials = 5
  Clique 15: Mean Accuracy = 0.9076 ± 0.0036, Trials = 5
  Clique 16: Mean Accuracy = 0.8939 ± 0.0093, Trials = 5
  Clique 17: Mean Accuracy = 0.8155 ± 0.1471, Trials = 5
  Clique 18: Mean Accuracy = 0.8916 ± 0.0058, Trials = 5
  Clique 19: Mean Accuracy = 0.8045 ± 0.1332, Trials = 5
  Clique 24: Mean Accuracy = 0.7436 ± 0.0377, Trials = 5
  Clique 26: Mean Accuracy = 0.8323 ± 0.0054, Trials = 5
  Clique 30: Mean Accuracy = 0.7471 ± 0.0893, Trials = 5
  Clique 34: Mean Accuracy = 0.7099 ± 0.0844, Trials = 5
  Clique 35: Mean Accuracy = 0.6604 ± 0.1456, Trials = 5
  Clique 38: Mean Accuracy = 0.7517 ± 0.0080, Trials = 5
  Clique 40: Mean Accuracy = 0.8341 ± 0.1798, Trials = 5
  Clique 44: Mean Accuracy = 0.9936 ± 0.0017, Trials = 5
  Cliq